# BART-large CNN — DIMER abstractive summarization tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/bart-cnn-summarization-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/bart-cnn-summarization-pipeline/blob/main/tutorials/bart_summarization_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-facebook%2Fbart--large--cnn-ffcc4d?style=flat)](https://huggingface.co/facebook/bart-large-cnn) [![Upstream](https://img.shields.io/badge/Upstream-facebookresearch%2Ffairseq-181717?style=flat&logo=github&logoColor=white)](https://github.com/facebookresearch/fairseq/tree/main/examples/bart) [![arXiv](https://img.shields.io/badge/arXiv-1910.13461-b31b1b.svg)](https://arxiv.org/abs/1910.13461)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** abstractive summarization of one English document by deterministic beam search (pinned generation defaults; token counts and a truncation flag reported; no score) using the pinned BART-large CNN weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/bart_summarization_pipeline/pipeline.py` at revision `7c424b79c6ae`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `37f520fa929c961707657b28798b30c003dd100b` (~1628 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the byte-level BPE tokenizer encodes the whole document once, the 12-layer bidirectional encoder of the 406 M-parameter BART-large reads it, and the 12-layer autoregressive decoder — fine-tuned upstream on CNN/DailyMail article/highlight pairs — writes a summary token by token under **deterministic beam search**: `num_beams` 4, `length_penalty` 2.0, `no_repeat_ngram_size` 3, `early_stopping`, and the length bounds of the snapshot's `generation_config_for_summarization.json` (`max_length` 142 / `min_length` 56, exposed as `DEFAULT_MAX_NEW_TOKENS` 141 / `DEFAULT_MIN_NEW_TOKENS` 55 because the upstream bounds count the decoder start token). **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. What the upstream checkpoint supplies is the encoder-decoder, the language-model head, the generation config and the tokenizer; what the carried pipeline module adds is manifest verification, input and generation-setting validation against named ceilings (over-long documents are rejected, not truncated or chunked), a fixed output contract that reports `generated_tokens`, `input_tokens`, `truncated` and `stopped_by`, and the `validate_inputs` and `evaluation_report` stage helpers. **The pipeline emits no score, probability or quality metric** — a summary is free text, and the repository ships no ROUGE helper.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, author one synthetic three-paragraph passage (or upload your own document), stage and digest-verify the immutable upstream snapshot, surface the pipeline's ceilings and the pinned generation defaults and validate the input into one input manifest, run `summarize` with the pinned defaults and once more with a shorter length bound and read the token counts and truncation flag correctly, read from the machine-readable evaluation report why no metric is reported and what reference data would make the task measurable, and export the summaries alongside their settings plus provenance.

**This notebook does not demonstrate:** extractive summarization with guaranteed source spans, multi-document or long-document (chunked) summarization, headline generation, sampling-based or diverse decoding, fine-tuning, classification or question answering (the `bart-mnli-zero-shot-classification-pipeline` sibling covers zero-shot classification), non-English text, or any faithfulness or quality score. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (also float32; the pipeline loads the checkpoint in float32 on both). This is a 406 M-parameter model: the model card's CPU smoke loaded and verified the 1.63 GB snapshot in 6.72 s and summarised a 232-token passage with 4 beams in 4.28 s, so the default path runs in about a minute on a hosted CPU runtime once the ~1.6 GB `model.safetensors` download has finished. The pinned `torch==2.14.0` install and that download are the largest transfers of the run; allow ~2 GB of free RAM for the weights.
- **Knowledge:** basic Python; what beam search over a decoder does and why it is deterministic; why an abstractive summary can contain statements the source does not (hallucination) and why ROUGE against references is needed to measure quality.
- **Data:** the default sample is one synthetic three-paragraph passage authored in code (a fictional town-council report), so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one UTF-8 text file holding the document to summarise (at most `MAX_TEXT_CHARS` characters and `MAX_INPUT_TOKENS` BPE tokens; longer documents are rejected, not truncated). Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `facebook/bart-large-cnn` snapshot (~1628 MB in total) at revision `37f520fa929c…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'bart-cnn-summarization-pipeline',
    'repository_revision': '7c424b79c6aea88a97448cb5ba168e50a019d6b5',
    'embedded_module': 'src/bart_summarization_pipeline/pipeline.py',
    'embedded_modules': ['src/bart_summarization_pipeline/pipeline.py'],
    'module_sha256': '751eeae1ce75bec4dfcca22336dff9d5909cf6f449cecdbce1a403d007c90b70',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/bart_summarization_pipeline/` @ `7c424b79c6ae`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/bart_summarization_pipeline/pipeline.py`

In [ ]:
"""Abstractive summarization with the pinned ``facebook/bart-large-cnn`` checkpoint.

Weights load only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed,
from the Hugging Face Hub at the pinned revision. One task method, ``summarize``: deterministic beam search
whose defaults are the values in the snapshot's ``generation_config_for_summarization.json``.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

MODEL_ID = "facebook/bart-large-cnn"
MODEL_REVISION = "37f520fa929c961707657b28798b30c003dd100b"
MODEL_LICENSE = "mit"
MODEL_KEY = "bart-large-cnn"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Ceilings. 1024 is max_position_embeddings in the snapshot config.json; longer inputs are rejected, not cut.
MAX_INPUT_TOKENS = 1024
MAX_TEXT_CHARS = 40_000  # pre-tokenisation guard on the input string; ~4 chars per BPE token on English text
MAX_NEW_TOKENS = 512  # ceiling on decoder steps per call
MAX_NUM_BEAMS = 8
MAX_NO_REPEAT_NGRAM_SIZE = 10
LENGTH_PENALTY_RANGE = (-5.0, 5.0)
# Defaults read from the snapshot's generation_config_for_summarization.json (identical to
# generation_config.json): num_beams 4, length_penalty 2.0, no_repeat_ngram_size 3, early_stopping true,
# max_length 142, min_length 56. Upstream's max_length/min_length count the decoder start token, so the
# equivalent *new*-token bounds are one lower.
DEFAULT_NUM_BEAMS = 4
DEFAULT_LENGTH_PENALTY = 2.0
DEFAULT_NO_REPEAT_NGRAM_SIZE = 3
DEFAULT_MAX_NEW_TOKENS = 141  # pinned max_length 142 - 1
DEFAULT_MIN_NEW_TOKENS = 55  # pinned min_length 56 - 1
EARLY_STOPPING = True
DECISION_RULE = (
    "deterministic beam search (do_sample=False, early_stopping=True): the highest length-penalised "
    "log-probability beam is returned; no sampling, no seed"
)
GENERATION_CONFIG_FILE = "generation_config_for_summarization.json"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _check_settings(
    max_new_tokens: Any, min_new_tokens: Any, num_beams: Any, length_penalty: Any, no_repeat_ngram_size: Any
) -> dict[str, Any]:
    """Raise TypeError/ValueError naming the first violated generation ceiling; return the settings."""
    for name, value, low, high in (
        ("max_new_tokens", max_new_tokens, 1, MAX_NEW_TOKENS),
        ("min_new_tokens", min_new_tokens, 0, MAX_NEW_TOKENS),
        ("num_beams", num_beams, 1, MAX_NUM_BEAMS),
        ("no_repeat_ngram_size", no_repeat_ngram_size, 0, MAX_NO_REPEAT_NGRAM_SIZE),
    ):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{name} must be an int")
        if not low <= value <= high:
            raise ValueError(f"{name} must be between {low} and {high}, got {value}")
    if min_new_tokens > max_new_tokens:
        raise ValueError(f"min_new_tokens {min_new_tokens} exceeds max_new_tokens {max_new_tokens}")
    if isinstance(length_penalty, bool) or not isinstance(length_penalty, int | float):
        raise TypeError("length_penalty must be a float")
    if not LENGTH_PENALTY_RANGE[0] <= length_penalty <= LENGTH_PENALTY_RANGE[1]:
        raise ValueError(f"length_penalty must be within {LENGTH_PENALTY_RANGE}, got {length_penalty}")
    return {
        "max_new_tokens": max_new_tokens,
        "min_new_tokens": min_new_tokens,
        "num_beams": num_beams,
        "length_penalty": float(length_penalty),
        "no_repeat_ngram_size": no_repeat_ngram_size,
        "early_stopping": EARLY_STOPPING,
        "do_sample": False,
        "decision_rule": DECISION_RULE,
    }


def _check_text(text: Any, name: str = "text") -> str:
    if not isinstance(text, str):
        raise TypeError(f"{name} must be str, got {type(text).__name__}")
    if not text.strip():
        raise ValueError(f"{name} is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"{name} has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
    return text


def _check_input_tokens(n_input: int) -> int:
    """The encoder-token ceiling, applied once the tokenizer has counted."""
    if n_input > MAX_INPUT_TOKENS:
        raise ValueError(f"input is {n_input} tokens; ceiling is MAX_INPUT_TOKENS={MAX_INPUT_TOKENS}")
    return n_input


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one non-empty str: the document to summarise (English news-style prose)",
    "text_chars": [1, MAX_TEXT_CHARS],
    "input_tokens": [1, MAX_INPUT_TOKENS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "min_new_tokens": [0, MAX_NEW_TOKENS],
    "num_beams": [1, MAX_NUM_BEAMS],
    "length_penalty": list(LENGTH_PENALTY_RANGE),
    "no_repeat_ngram_size": [0, MAX_NO_REPEAT_NGRAM_SIZE],
    "defaults_from": GENERATION_CONFIG_FILE,
    "decision_rule": DECISION_RULE,
    "preprocessing": (
        "byte-level BPE encoding with <s>/</s> added and no truncation: an input over MAX_INPUT_TOKENS is "
        "rejected with a ValueError naming the count, never cut"
    ),
}


def validate_inputs(
    texts: Sequence[str],
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    min_new_tokens: int = DEFAULT_MIN_NEW_TOKENS,
    num_beams: int = DEFAULT_NUM_BEAMS,
    length_penalty: float = DEFAULT_LENGTH_PENALTY,
    no_repeat_ngram_size: int = DEFAULT_NO_REPEAT_NGRAM_SIZE,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``summarize`` would: both route through ``_check_text``
    and ``_check_settings``. ``summarize`` takes one text per call, so ``texts`` is the batch the notebook
    will loop over and every entry is validated with the same settings. The encoder-token ceiling
    (``MAX_INPUT_TOKENS``) needs the loaded tokenizer and is enforced inside ``summarize``.
    """
    if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a sequence of str, not a single string")
    if not texts:
        raise ValueError("texts must hold at least one item")
    checked = [_check_text(text, f"texts[{i}]") for i, text in enumerate(texts)]
    settings = _check_settings(
        max_new_tokens, min_new_tokens, num_beams, length_penalty, no_repeat_ngram_size
    )
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per text")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[i] if names else f"doc-{i}",
                "chars": len(text),
                "paragraphs": len(text.split("\n\n")),
            }
            for i, text in enumerate(checked)
        ],
        "generation": settings,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], references: Sequence[str] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though no metric exists here.

    The repository ships no ROUGE (or any) metric helper, so the verdict is always ``not-measurable``
    (EVAL9). ``references`` exists for interface parity with the fleet's other pipelines and is recorded in
    ``reason`` rather than scored: ROUGE needs a scorer and enough referenced documents to state a
    dispersion, and a proxy such as compression ratio or copy rate would misrepresent a plumbing check
    as a quality measurement.
    """
    generation = result.get("generation", {})
    supplied = references is not None
    return {
        "task": "abstractive summarization of one English document (CNN/DailyMail fine-tune)",
        "score_semantics": (
            "the pipeline emits no probability, confidence or score: generated_tokens, input_tokens, "
            "truncated and stopped_by are counts and flags, and "
            f"{generation.get('decision_rule', DECISION_RULE)} produces some token at every step with no "
            "minimum-probability cut-off and no shipped acceptance threshold"
        ),
        "sample_kind": sample_kind,
        "n_generated_tokens": int(result.get("generated_tokens", 0)),
        "truncated": bool(result.get("truncated", False)),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "the repository ships no ROUGE or other metric helper and a summary has no ground truth here"
            + (
                "; reference summaries were supplied but no metric helper exists to score them, and one "
                "reference is not a dispersion"
                if supplied
                else "; the evaluated sample has no reference summary"
            )
        ),
        "needs": (
            "one or more reference summaries per document from the deployment domain over enough documents "
            "to state a dispersion, scored with the caller's own ROUGE-1/2/L implementation (the upstream "
            "card reports ROUGE on CNN/DailyMail; nothing here reproduces it), plus a faithfulness check "
            "against the source, excluding or re-running outputs whose truncated flag is true; no proxy such "
            "as compression ratio or copy rate substitutes for that"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class BARTSummarizationPipeline:
    """``_runner(text, settings)`` -> ``(summary_text, generated_tokens, stopped_by)``;
    ``_count_tokens(text)`` -> encoder token count incl. <s>/</s>. Both injectable so tests run offline."""

    _runner: Callable[[str, dict[str, Any]], tuple[str, int, str]]
    _count_tokens: Callable[[str], int]
    device: str = "cpu"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BARTSummarizationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, kwargs, source = str(root), {"local_files_only": True}, "local-snapshot"
        elif allow_download:
            location, kwargs, source = MODEL_ID, {}, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import BartForConditionalGeneration, BartTokenizerFast

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = BartTokenizerFast.from_pretrained(
            location, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = BartForConditionalGeneration.from_pretrained(
            location, revision=MODEL_REVISION, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()
        special = {tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id}

        def count_tokens(text: str) -> int:
            return len(tokenizer(text, truncation=False)["input_ids"])

        def runner(text: str, settings: dict[str, Any]) -> tuple[str, int, str]:
            enc = tokenizer(text, return_tensors="pt", truncation=False).to(resolved_device)
            with torch.inference_mode():
                out = model.generate(
                    **enc,
                    max_new_tokens=settings["max_new_tokens"],
                    min_new_tokens=settings["min_new_tokens"],
                    num_beams=settings["num_beams"],
                    length_penalty=settings["length_penalty"],
                    no_repeat_ngram_size=settings["no_repeat_ngram_size"],
                    early_stopping=settings["early_stopping"],
                    do_sample=False,
                )
            ids = out[0].tolist()
            content = [t for t in ids[1:] if t not in special]  # ids[0] is the decoder start token
            stopped_by = "eos" if tokenizer.eos_token_id in ids[1:] else "max_new_tokens"
            return tokenizer.decode(content, skip_special_tokens=True).strip(), len(content), stopped_by

        return cls(runner, count_tokens, resolved_device, source)

    def summarize(
        self,
        text: str,
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        min_new_tokens: int = DEFAULT_MIN_NEW_TOKENS,
        num_beams: int = DEFAULT_NUM_BEAMS,
        length_penalty: float = DEFAULT_LENGTH_PENALTY,
        no_repeat_ngram_size: int = DEFAULT_NO_REPEAT_NGRAM_SIZE,
    ) -> dict[str, Any]:
        """Summarise one document by deterministic beam search; defaults are the pinned generation config."""
        text = _check_text(text)
        settings = _check_settings(
            max_new_tokens, min_new_tokens, num_beams, length_penalty, no_repeat_ngram_size
        )
        n_input = _check_input_tokens(self._count_tokens(text))
        summary, n_generated, stopped_by = self._runner(text, settings)
        if not (isinstance(summary, str) and isinstance(n_generated, int) and isinstance(stopped_by, str)):
            raise RuntimeError("runner must return (str, int, str)")
        return {
            "summary": summary,
            "generated_tokens": n_generated,
            "input_tokens": n_input,
            "truncated": stopped_by != "eos",
            "stopped_by": stopped_by,
            "generation": settings,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `37f520fa929c…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `BARTSummarizationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "bart-large-cnn",
  "modelId": "facebook/bart-large-cnn",
  "revision": "37f520fa929c961707657b28798b30c003dd100b",
  "files": [
    {
      "path": "README.md",
      "bytes": 6009,
      "sha256": "a49b4b7fa5e64a05277d57cb767d9b8e8d4789a09cab9647beadceaaed86e170"
    },
    {
      "path": "config.json",
      "bytes": 1585,
      "sha256": "c6cb642aec929b65f514ee0ec7c04f9de19f705c143491577ecd8b7cc923c6ed"
    },
    {
      "path": "generation_config.json",
      "bytes": 363,
      "sha256": "4897361917410254e3132e8fe7786d37f3ef7cff54a650845c1147c2450a790f"
    },
    {
      "path": "generation_config_for_summarization.json",
      "bytes": 363,
      "sha256": "4897361917410254e3132e8fe7786d37f3ef7cff54a650845c1147c2450a790f"
    },
    {
      "path": "merges.txt",
      "bytes": 456318,
      "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1625222120,
      "sha256": "40041830399afb5348525ef8354b007ecec4286fdf3524f7e6b54377e17096cb"
    },
    {
      "path": "tokenizer.json",
      "bytes": 1355863,
      "sha256": "847bbeab6174d66a88898f729d52fa8d355fafe1bea101cf960dd404581df70e"
    },
    {
      "path": "vocab.json",
      "bytes": 898823,
      "sha256": "9e7f63c2d15d666b52e21d250d2e513b87c9b713cfa6987a82ed89e5e6e50655"
    }
  ],
  "totalBytes": 1627941444
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = BARTSummarizationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Author the synthetic sample or optional BYOD

The default sample is **synthetic**, written in this cell: a fictional three-paragraph report about a town council approving a water-mains project (the model card's smoke passage), given the identifier `doc`. It is prose in the expository register the CNN/DailyMail fine-tune expects, but it is not news, carries no reference summary, and proves nothing about quality — a run on it is a plumbing check, never benchmark evidence. `SUMMARY_MAX_NEW_TOKENS` and `NUM_BEAMS` are Colab form parameters checked against the carried module in Section 5; their defaults are the pinned generation config.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 text file holding the document to summarise, at most `MAX_TEXT_CHARS` characters and at most `MAX_INPUT_TOKENS` BPE tokens including `<s>`/`</s>` (a longer document is rejected by the pipeline, not truncated or chunked). The upload stays inside this runtime. If you also hold a reference summary, keep it outside the notebook — Section 7 explains what to compute with it.

In [ ]:
import hashlib

USE_BYOD = False  # @param {type:"boolean"}
SUMMARY_MAX_NEW_TOKENS = 141  # @param {type:"integer"}
NUM_BEAMS = 4  # @param {type:"integer"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    document = uploaded[sample_name].decode('utf-8').strip()
    if not document:
        raise ValueError(f'{sample_name}: the uploaded file is empty')
    sample_kind = 'BYOD upload'
else:
    document = (
        'The town council of Millbrook voted on Tuesday evening to approve a three-year plan to replace the '
        'aging water mains beneath the historic district. The plan, which had been debated for more than a '
        'year, will cost an estimated 4.2 million dollars and is scheduled to begin in the spring. Council '
        'members said the decision was driven by a series of pipe failures last winter that left several '
        'streets without water for days.\n\n'
        'Under the approved schedule, crews will work one block at a time so that no more than two streets are '
        'closed on any given day. The public works director told residents that most of the disruption would '
        'fall in the first eighteen months, with paving and landscaping to follow. Businesses along Main Street '
        'will receive advance notice of closures and a dedicated contact for complaints.\n\n'
        'Funding will come from a combination of a state infrastructure grant and a modest increase in water '
        'rates, which the council set at three percent per year for the duration of the project. Two members '
        'voted against the rate increase, arguing that the grant alone should have covered the work, but the '
        'majority said delaying the project any further would only raise its cost.'
    )
    sample_name = 'synthetic_millbrook_council_report'
    sample_kind = 'synthetic (authored in this cell; the model card smoke passage)'
document_id = 'doc'
sample_sha256 = hashlib.sha256(document.encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'chars': len(document), 'paragraphs': len(document.split('\n\n')), 'max_new_tokens': SUMMARY_MAX_NEW_TOKENS, 'num_beams': NUM_BEAMS, 'text_sha256': sample_sha256})
print(document[:300] + (' …' if len(document) > 300 else ''))

## 5. Validate the input → input manifest

`validate_inputs` is the pipeline's public validation stage: it takes the batch of documents `summarize` will be called on (one here) and the generation settings, and runs the method's own checks — `_check_text` and `_check_settings` — so a rejection here is a rejection there. `MAX_TEXT_CHARS` is the character guard applied before tokenisation; `MAX_INPUT_TOKENS` (1024, the checkpoint's position limit) is applied after tokenisation and **rejects** longer documents rather than truncating or chunking them, so it is enforced inside the pipeline and cannot be observed at this stage; `MAX_NEW_TOKENS`, `MAX_NUM_BEAMS`, `LENGTH_PENALTY_RANGE` and `MAX_NO_REPEAT_NGRAM_SIZE` bound the generation settings, whose defaults (`DEFAULT_*`) are the pinned `generation_config_for_summarization.json`; `DECISION_RULE` names the decoding rule. The manifest records the exact settings that will be used and is written to `outputs/bart_summarization_input_manifest.json`. To show what rejection looks like, the cell also validates with `num_beams` one above the ceiling and records the pipeline's own error message as a finding. The notebook never trims or alters the document.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
ceilings = {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_INPUT_TOKENS': MAX_INPUT_TOKENS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'MAX_NUM_BEAMS': MAX_NUM_BEAMS, 'LENGTH_PENALTY_RANGE': LENGTH_PENALTY_RANGE, 'MAX_NO_REPEAT_NGRAM_SIZE': MAX_NO_REPEAT_NGRAM_SIZE}
print(ceilings)
print({'defaults_from': GENERATION_CONFIG_FILE, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DEFAULT_MIN_NEW_TOKENS': DEFAULT_MIN_NEW_TOKENS, 'DEFAULT_NUM_BEAMS': DEFAULT_NUM_BEAMS, 'DEFAULT_LENGTH_PENALTY': DEFAULT_LENGTH_PENALTY, 'DEFAULT_NO_REPEAT_NGRAM_SIZE': DEFAULT_NO_REPEAT_NGRAM_SIZE, 'EARLY_STOPPING': EARLY_STOPPING, 'DECISION_RULE': DECISION_RULE})
input_manifest = validate_inputs([document], max_new_tokens=SUMMARY_MAX_NEW_TOKENS, num_beams=NUM_BEAMS, names=[document_id])
# Demonstrate a ceiling rejection; the finding is recorded, not swallowed.
try:
    validate_inputs([document], max_new_tokens=SUMMARY_MAX_NEW_TOKENS, num_beams=MAX_NUM_BEAMS + 1)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'num-beams-ceiling-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/bart_summarization_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))
print({'token_ceiling': f'MAX_INPUT_TOKENS={MAX_INPUT_TOKENS} is checked by the pipeline after tokenisation and rejects, never truncates or chunks'})

## 6. Summarize and read the output correctly

**Input/output contract.** `summarize(text, max_new_tokens=..., min_new_tokens=..., num_beams=..., length_penalty=..., no_repeat_ngram_size=...)` takes one string and returns `summary` (free text), `generated_tokens` (content tokens emitted, excluding `<s>`/`</s>`/padding), `input_tokens` (encoder tokens including `<s>`/`</s>`), `truncated` (`True` when the decoder hit `max_new_tokens` before emitting end-of-sequence), `stopped_by` (`eos` or `max_new_tokens`), the full `generation` settings with the `decision_rule`, the device and the model identity. **Decoding semantics:** deterministic beam search — `num_beams` partial summaries are kept at each step and the finished beam with the highest length-penalised log-probability is returned; `do_sample` is fixed to `False`, so the same document on the same device gives the same summary and no seed is needed; `min_new_tokens` is a floor that forbids end-of-sequence before that many tokens, which on a short input forces the model to keep writing. **No score is emitted:** nothing in the result says whether the summary is faithful or good, and the pipeline applies no threshold. The model card's CPU smoke on this passage produced 81 tokens, stopped by `eos`, reproducing the first two source sentences verbatim and paraphrasing the third — one observation of the model's largely extractive behaviour on short expository input, not an expected value; CPU and CUDA kernels can pick different beams when two are close. The cell runs the pinned defaults and then a shorter bound (`max_new_tokens` 60, `min_new_tokens` 20) so the effect of the length settings is visible.

In [ ]:
import time

started = time.perf_counter()
result = pipe.summarize(document, max_new_tokens=SUMMARY_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
summarize_elapsed = time.perf_counter() - started
checks = {
    'summary_is_non_empty_text': isinstance(result['summary'], str) and bool(result['summary'].strip()),
    'generated_within_bound': 1 <= result['generated_tokens'] <= SUMMARY_MAX_NEW_TOKENS <= MAX_NEW_TOKENS,
    'input_within_ceiling': 1 <= result['input_tokens'] <= MAX_INPUT_TOKENS,
    'truncated_matches_stopped_by': result['truncated'] == (result['stopped_by'] != 'eos'),
    'settings_echoed': result['generation']['max_new_tokens'] == SUMMARY_MAX_NEW_TOKENS and result['generation']['num_beams'] == NUM_BEAMS,
    'deterministic_decoding': result['generation']['do_sample'] is False,
}
if not all(checks.values()):
    raise RuntimeError(f'summarize output failed a sanity check: {checks}')
print({key: value for key, value in result.items() if key != 'summary'})
print({'seconds': round(summarize_elapsed, 3), 'checks': checks, 'no_score': 'the pipeline emits no probability or quality score; truncated is a length flag'})
print('summary (pinned defaults):')
print(result['summary'])
started = time.perf_counter()
short_result = pipe.summarize(document, max_new_tokens=60, min_new_tokens=20, num_beams=NUM_BEAMS)
short_elapsed = time.perf_counter() - started
print({'short_run': {key: short_result[key] for key in ('generated_tokens', 'truncated', 'stopped_by')}, 'seconds': round(short_elapsed, 3)})
print('summary (max_new_tokens=60, min_new_tokens=20):')
print(short_result['summary'])

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report — here, one that says nothing is measurable. The repository ships **no ROUGE or other metric helper and reports no performance measure**, so the verdict is always `not-measurable` and the report states what would make the task measurable: one or more reference summaries per document from the deployment domain over enough documents to state a dispersion, scored with the caller's own ROUGE-1/2/L implementation, plus a faithfulness check against the source (ROUGE rewards overlap, not truth), excluding or re-running outputs whose `truncated` flag is true. Supplying a reference does **not** change the verdict — one reference is not a dispersion and no scorer is shipped — so the helper records that in `reason` instead. The upstream card's ROUGE numbers on CNN/DailyMail are upstream-reported and nothing here reproduces them. The sanity checks printed in Section 6 remain falsifiable plumbing checks, not results. The report is written to `outputs/bart_summarization_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, None, sample_kind=sample_kind)
with open('outputs/bart_summarization_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No metric is reported: the repository ships no ROUGE helper and the sample carries no reference summary; score your own references with your own ROUGE implementation and check faithfulness against the source.')

## 8. Export summaries alongside their settings, and provenance

Two further files are written under `outputs/` beside the input manifest and the evaluation report. The summaries go to CSV (`outputs/bart_summarization_summaries.csv`) with one row per run — `document_id`, `run`, `max_new_tokens`, `min_new_tokens`, `num_beams`, `length_penalty`, `no_repeat_ngram_size`, `generated_tokens`, `input_tokens`, `truncated`, `stopped_by`, `summary` — so every summary stays attached to the settings that produced it. One JSON record (`outputs/bart_summarization_result.json`) preserves both results in full (summary, counts, flags, generation settings, decision rule, seconds), the sanity checks, the ceilings in force, the input manifest, the evaluation report, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, the verified snapshot summary, and the runtime identity (Python, `torch`, `transformers`, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
import csv

runs = [('pinned-defaults', result, summarize_elapsed), ('short-bound', short_result, short_elapsed)]
with open('outputs/bart_summarization_summaries.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['document_id', 'run', 'max_new_tokens', 'min_new_tokens', 'num_beams', 'length_penalty', 'no_repeat_ngram_size', 'generated_tokens', 'input_tokens', 'truncated', 'stopped_by', 'summary'])
    for run_name, run_result, _ in runs:
        generation = run_result['generation']
        writer.writerow([document_id, run_name, generation['max_new_tokens'], generation['min_new_tokens'], generation['num_beams'], generation['length_penalty'], generation['no_repeat_ngram_size'], run_result['generated_tokens'], run_result['input_tokens'], run_result['truncated'], run_result['stopped_by'], run_result['summary']])
payload = {
    'document_id': document_id,
    'runs': [
        {'run': run_name, 'seconds': round(seconds, 3), **{key: value for key, value in run_result.items() if key not in ('device', 'source', 'model_id', 'model_revision')}}
        for run_name, run_result, seconds in runs
    ],
    'decoding': 'deterministic beam search (do_sample=False); no score, probability or quality metric is emitted; truncated is a length flag; no threshold shipped',
    'sanity_checks': checks,
    'summaries_file': 'outputs/bart_summarization_summaries.csv',
    'ceilings': ceilings,
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'sample': {'name': sample_name, 'kind': sample_kind, 'chars': len(document), 'text_sha256': sample_sha256},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': snapshot['path'], 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes')},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'dtype': 'float32',
    },
}
with open('outputs/bart_summarization_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The summaries are the output of deterministic beam search over a decoder fine-tuned on news highlights: the highest length-penalised beam is returned, some token is emitted at every step, `min_new_tokens` forces a minimum length, and nothing in the result says whether a sentence is faithful to the source — an abstractive summarizer can state what the document does not and drop what matters most. The pipeline emits no score and applies no threshold; the caller owns any acceptance rule and must set it on reference summaries from their own domain. On the synthetic sample the run is plumbing evidence only; the evaluation report is `not-measurable` because no metric can be computed without references and a scorer, and a real evaluation needs ROUGE against references over enough documents to state a dispersion plus a faithfulness check the repository does not ship. Documents over 1024 BPE tokens are rejected, not truncated or chunked; the checkpoint is English only, writes in the news-highlight register whatever the input, and carries whatever associations CNN/DailyMail and the BART pre-training corpus contain, which neither the upstream card nor this repository has audited; the pipeline exposes no sampling, no fine-tuning and no logits. Inference is deterministic on a fixed device and dtype, but CPU and CUDA kernels can pick different beams when two are close.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model snapshot, validate the demonstrated input and settings against the enforced ceilings, execute the public `summarize` path under two length bounds, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, summary quality or faithfulness on any domain, a usable acceptance rule, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments.** Lower `SUMMARY_MAX_NEW_TOKENS` until `truncated` turns true and read how the summary ends mid-thought; set `NUM_BEAMS` to 1 (greedy) and compare the wording; paste a document whose facts you know and count the statements the summary makes that the source does not — that is the faithfulness check the evaluation report asks for; assemble a dozen documents with reference summaries of your own and compute ROUGE with a bootstrap interval. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: https://github.com/kurtvalcorza/bart-cnn-summarization-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/bart-cnn-summarization-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/bart-cnn-summarization-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/facebook/bart-large-cnn
- Upstream code: https://github.com/facebookresearch/fairseq/tree/main/examples/bart
- BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension: https://arxiv.org/abs/1910.13461
- Get To The Point: Summarization with Pointer-Generator Networks (the CNN/DailyMail summarization split): https://arxiv.org/abs/1704.04368